# Data Collection

## Imports & Configuration

This cell is just to set up imports, and the main things needed for the data collection, so the mendoza basin coordinates, and then also the locations of data directories.

In [3]:
import cdsapi
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import zipfile
import io

# Mendoza Basin Coordinates
basin = {
    "north": -32,
    "south": -34.5,
    "west": -70.5,
    "east": -68,    
}

RAW_DIR = "../data/raw"
PRO_DIR = "../data/processed"

## Downloading from CDS

This cell is for downloading the files from Copornecius website, using the cdsapi. I chunk the data into groups of 5 years for ease of downloading. I pulled the variables for the reasons below:

**total_precipitation** - The basis of the basin analysis, includes snow and rainfall precipitation and is used for SPI, water balance, and also ML training data.  
**2m_temperature** - Help with ML features, causes for loss of water (evaporation) as well as plant needs etc. Also to explore the trend of if the basin is warming.    
**potential_evaporation** - Used in SPEI calculations and also water deficit calculations.  
**runoff** - Used to determine whether water returns to the ground or to the ocean.  
**volumetric_soil_water_layer_1** - Further ML refinement and also anomaly detection, to see the ground condition. 

In [4]:
output_file = os.path.join(RAW_DIR, "era5_mendoza_monthly.nc")

c = cdsapi.Client()

chunk_size = 5
year_range = list(range(1980, 2025))
chunk_files = []

for start in range(0, len(year_range), chunk_size):
    year_chunk = year_range[start:start + chunk_size]
    chunk_file = os.path.join(RAW_DIR, f"era5_mendoza_{year_chunk[0]}_{year_chunk[-1]}.nc")
    chunk_files.append(chunk_file)

    if os.path.exists(chunk_file):
        print(f"File {chunk_file} already exists.")
        continue

    print(f"Downloading {year_chunk[0]}-{year_chunk[-1]}...")
    c.retrieve(
        "reanalysis-era5-single-levels-monthly-means",
        {
            "product_type": "monthly_averaged_reanalysis",
            "variable": [
                "total_precipitation",
                "2m_temperature",
                "potential_evaporation",
                "runoff",
                "volumetric_soil_water_layer_1",
            ],
            "year": [str(y) for y in year_chunk],
            "month": [str(m).zfill(2) for m in range(1, 13)],
            "time": "00:00",
            "area": [basin["north"], basin["west"], basin["south"], basin["east"]],
            "format": "netcdf",
        },
        chunk_file,
    )
print("All downloads complete.")

File ../data/raw\era5_mendoza_1980_1984.nc already exists.
File ../data/raw\era5_mendoza_1985_1989.nc already exists.
File ../data/raw\era5_mendoza_1990_1994.nc already exists.
File ../data/raw\era5_mendoza_1995_1999.nc already exists.
File ../data/raw\era5_mendoza_2000_2004.nc already exists.
File ../data/raw\era5_mendoza_2005_2009.nc already exists.
File ../data/raw\era5_mendoza_2010_2014.nc already exists.
File ../data/raw\era5_mendoza_2015_2019.nc already exists.
File ../data/raw\era5_mendoza_2020_2024.nc already exists.
All downloads complete.


## Read ZIPs into Dataframes

I now read the zipped files created, and extract the spatial averages as dataframes. 

**data_stream-moda_stepType-avgua.nc**: Includes the state variables (temp and soil moisture)   
**data_stream-moda_stepType-avgad.nc**: Has all the hydrological variables, so precipitation, evaporation, and runoff

In [5]:
zip_files = sorted([
    f for f in os.listdir(RAW_DIR)
    if f.startswith("era5_mendoza_") and f.endswith(".nc")
    and f != "era5_mendoza_monthly.nc"
])
print(f"Found {len(zip_files)} zip files\n")

avgua_frames = []
avgad_frames = []

for zf_name in zip_files:
    zf_path = os.path.join(RAW_DIR, zf_name)
    
    with zipfile.ZipFile(zf_path, "r") as zf:
        for info in zf.infolist():
            file_bytes = zf.read(info.filename)
            ds = xr.open_dataset(io.BytesIO(file_bytes), engine="h5netcdf")
        
            if "valid_time" in ds.dims:
                ds = ds.rename({"valid_time": "time"})
            
            for drop_var in ["number", "expver"]:
                if drop_var in ds.dims:
                    ds = ds.isel({drop_var: 0})
                elif drop_var in ds:
                    ds = ds.drop_vars(drop_var)
            
            frame = ds.mean(dim=["latitude", "longitude"]).to_dataframe()
            
            if "avgua" in info.filename:
                avgua_frames.append(frame)
            elif "avgad" in info.filename:
                avgad_frames.append(frame)
            
            ds.close()
    
    print(f"Read: {zf_name}")

print(f"\nCollected {len(avgua_frames)} avgua chunks (temp, soil moisture)")
print(f"Collected {len(avgad_frames)} avgad chunks (precip, evap, runoff)")

Found 9 zip files

Read: era5_mendoza_1980_1984.nc
Read: era5_mendoza_1985_1989.nc
Read: era5_mendoza_1990_1994.nc
Read: era5_mendoza_1995_1999.nc
Read: era5_mendoza_2000_2004.nc
Read: era5_mendoza_2005_2009.nc
Read: era5_mendoza_2010_2014.nc
Read: era5_mendoza_2015_2019.nc
Read: era5_mendoza_2020_2024.nc

Collected 9 avgua chunks (temp, soil moisture)
Collected 9 avgad chunks (precip, evap, runoff)


## Join and Convert Units

Here I concatenate the individual dataframes and join the state and hydrological variables into a single dataframe. I also convert the raw ERA5 units to more intuitive measurements: 

**Precipitation**: Converted from meters/day to mm/month (multiplied by 1000 and days in month)  
**PEV**: Converted from meters/day to mm/month (multiplied by 1000 and days in month)   
**Runoff**: Converted from meters/day to mm/month (multiplied by 1000 and days in month)    
**Temperature**: Converted from Kelvin to Celsius (subtract 273.15)      
**Soil Moisture**: Extracted directly from the first layer (swvl1)  

In [6]:
df_avgua = pd.concat(avgua_frames).sort_index()
df_avgad = pd.concat(avgad_frames).sort_index()

print(f"avgua: {len(df_avgua)} rows, columns: {list(df_avgua.columns)}")
print(f"avgad: {len(df_avgad)} rows, columns: {list(df_avgad.columns)}")

df_avgua.index = df_avgua.index.normalize()
df_avgad.index = df_avgad.index.normalize()

df_raw = df_avgua.join(df_avgad, how="outer")
df_raw.index.name = "time"

days_in_month = df_raw.index.days_in_month

df = pd.DataFrame(index=df_raw.index)
df.index.name = "time"
df["precip_mm"] = df_raw["tp"] * 1000 * days_in_month
df["temp_c"] = df_raw["t2m"] - 273.15
df["pev_mm"] = -df_raw["pev"] * 1000 * days_in_month
df["runoff_mm"] = df_raw["ro"] * 1000 * days_in_month
df["soil_moisture_0_7cm"] = df_raw["swvl1"]

df.head()

avgua: 540 rows, columns: ['t2m', 'swvl1']
avgad: 540 rows, columns: ['tp', 'pev', 'ro']


,precip_mm,temp_c,pev_mm,runoff_mm,soil_moisture_0_7cm
time,,,,,
1980-01-01,63.820649,15.302368,183.912286,98.165597,0.191278
1980-02-01,93.886005,13.928650,148.222182,36.150087,0.191747
1980-03-01,66.787061,14.917786,132.601721,29.331790,0.162673
1980-04-01,162.947316,6.125000,59.435077,35.352784,0.201941
1980-05-01,115.186830,3.713196,39.293950,28.664771,0.218384


# Quality Checks and Creating CSV

Before exporting, I perform some basic validation on the final dataframe:

Shape & Period: Confirm the expected number of months and variables
Missing Values: Check each column for null values
Sanity Checks: Verify no negative precipitation/PEV values, and that temperature and precipitation ranges are physically reasonable

Once validated, the dataframe is exported as **mendoza_basin_monthly.csv** for use in downstream analysis.

In [7]:
print(f"\nShape: {df.shape[0]} months x {df.shape[1]} variables")
print(f"Period: {df.index[0].strftime('%Y-%m')} to {df.index[-1].strftime('%Y-%m')}")

print(f"\nMissing Values per Variable:")
for col in df.columns:
    count = df[col].isnull().sum()
    status = "None" if count == 0 else f"{count} missing"
    print(f"    {col:25s} {status}")

print(f"\nSanity Checks:")
print(f"    Negative Precipitation:   {(df['precip_mm'] < 0).sum()}")
print(f"    Negative PEV:      {(df['pev_mm'] < 0).sum()}")
print(f"    Temp Range:        {df['temp_c'].min():.1f} to {df['temp_c'].max():.1f} C")
print(f"    Max Monthly Precipitation: {df['precip_mm'].max():.1f} mm")

output_csv = os.path.join(PRO_DIR, "mendoza_basin_monthly.csv")
df.to_csv(output_csv)
print("\nMendoza_basin_monthly.csv saved as csv")


Shape: 540 months x 5 variables
Period: 1980-01 to 2024-12

Missing Values per Variable:
    precip_mm                 None
    temp_c                    None
    pev_mm                    None
    runoff_mm                 None
    soil_moisture_0_7cm       None

Sanity Checks:
    Negative Precipitation:   0
    Negative PEV:      0
    Temp Range:        -3.2 to 18.3 C
    Max Monthly Precipitation: 294.9 mm

Mendoza_basin_monthly.csv saved as csv
